<a href="https://colab.research.google.com/github/Robin-Chetry/ml_material/blob/main/Data_Driven_Analysis_of_Material_Properties_Using_Linear_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Project Introduction

This project explores how **fundamental physical properties of elements are related to each other** using **machine learning techniques**. Specifically, we analyze whether **melting temperature** can be used to predict other important material properties such as **Young's Modulus, Lattice Constant, and the Coefficient of Linear Thermal Expansion (CTE)**.

### Objective
The main goal of this project is to **identify patterns and relationships between thermal and mechanical properties of elements**. By applying **linear regression**, we attempt to determine whether melting temperature can act as a simple predictor for other material characteristics.

### What We Are Analyzing
In this study, we examine three key relationships:

1. **Young’s Modulus vs. Melting Temperature**  
   To understand whether materials with higher melting temperatures also tend to be mechanically stiffer.

2. **Lattice Constant vs. Melting Temperature**  
   To investigate whether the size of the atomic crystal structure influences thermal stability.

3. **Coefficient of Thermal Expansion vs. Melting Temperature**  
   To explore how strongly bonded materials expand when temperature changes.

### Why This Analysis is Important
Understanding relationships between material properties is important in **materials science and engineering** because:

- It helps **predict unknown properties** of materials using known data.
- It allows researchers to **screen materials quickly** without expensive experiments.
- It provides insight into **atomic bonding and crystal structure behavior**.
- It demonstrates how **machine learning can assist materials discovery**.

### Approach
To perform this analysis:

- A dataset of elemental properties was collected using **materials science databases**.
- The data was divided into **training and testing sets**.
- A **linear regression model** was used to learn relationships between melting temperature and other properties.
- **Interactive plots** were created to visualize the model predictions and compare them with actual data.

### Key Idea
This project demonstrates that **data-driven methods can reveal physical trends in materials**, helping scientists better understand how **atomic bonding and thermal stability influence mechanical and structural properties**.


In [ ]:
!pip install -U scikit-learn
!pip install pymatgen
!pip install mendeleev

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.3/332.3 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 962.5/962.5 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 4.4 MB/s eta 0:00:00
  Created wheel for bibtexparser: filename=bibtexparser-1.4.4-py3-none-any.whl size=43609 sha256=166fb140144ac5f3f37915bfb8fa3e8ba2774fdb804f00f71950dea8fb366d7e
  Stored in directory: /root/.cache/pip/wheels/54/f8/e6/ecfceb6af875ddc5096bb3811795ac336f50371009a601454d
Successfully built bibtexparser
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
%matplotlib inline

## Libraries Used

### 🛠️ Materials Data
- **pymatgen** – Provides material properties (Young’s Modulus, Atomic Mass, CTE).
- **mendeleev** – Provides elemental properties (Melting Point, Lattice Constant, Specific Heat).

### 🧠 Machine Learning
- **numpy** – Formats and reshapes data for ML models.
- **sklearn** – Builds the **Linear Regression** model and evaluates accuracy.

### 📊 Visualization & Utilities
- **matplotlib** – Creates graphs to visualize relationships.
- **random** – Shuffles data to avoid bias.

### Workflow
**Collect Data → Format with numpy → Train model with sklearn → Visualize with matplotlib**

In [ ]:
import numpy as np
import pymatgen as pymat
import mendeleev as mendel
from sklearn import datasets, linear_model
from sklearn.metrics import mean_squared_error, r2_score
from random import shuffle
import matplotlib.pyplot as plt

## Step 2: Data Acquisition – Key Points

### 1. Element Organization
- Elements are grouped by **crystal structure**: FCC, BCC, HCP, and Others.
- This ensures **balanced representation** of different atomic structures.
- The combined list of elements is **shuffled** to avoid bias in the dataset.

### 2. Data Extraction
For each element, properties are collected using libraries:

- **pymatgen**
  - Young’s Modulus
  - Coefficient of Thermal Expansion (CTE)

- **mendeleev**
  - Melting Point
  - Lattice Constant

The values are stored in lists (data buckets) for building the dataset.

### 3. Physical Hypothesis
- **Melting Point** → energy needed to break atomic bonds.
- **Young’s Modulus** → stiffness of those bonds.
- Hypothesis: **Higher melting point → higher Young’s Modulus**.

### 4. Data Validation
- Some elements may return **None values** if data is missing.
- ML models **cannot process missing values**, so such elements should be filtered.

### 5. Why Manual Grouping?
- **Balanced dataset** across crystal structures.
- **Filter out gases, liquids, and unstable elements**.
- Handle **allotropes** (elements with multiple crystal forms).

### 6. Why Collect Extra Properties?
Extra features help with deeper analysis and future models.

| Property | Purpose |
|--------|--------|
| Lattice Constant | Shows atomic spacing and bond geometry |
| CTE | Indicates how bonds respond to heat |
| Specific Heat | Energy required for atomic vibrations |

These features allow future **multi-variable ML models**.

### Summary Workflow
**Select elements → Extract properties → Validate data → Build dataset for ML**

In [ ]:
fcc_elements = ["Ag", "Al", "Au", "Cu", "Ir", "Ni", "Pb", "Pd", "Pt", "Rh", "Th", "Yb"]
bcc_elements = ["Ba", "Cr", "Eu", "Fe", "Li", "Mn", "Mo", "Na", "Nb", "Ta", "V", "W" ]
hcp_elements = ["Be", "Ca", "Cd", "Co", "Dy", "Er", "Gd", "Hf", "Ho", "Lu", "Mg", "Re",
                "Ru", "Sc", "Tb", "Ti", "Tl", "Tm", "Y", "Zn", "Zr"]
others = ["Sb", "Sm", "Bi", "Ce", "Sn", "Si"]
# Others (Solids): "Sb", "Sm", Bi" and "As" are Rhombohedral; "C" , "Ce" and "Sn" are Allotropic;
# "Si" and "Ge" are Face-centered diamond-cubic;

elements = fcc_elements + bcc_elements + hcp_elements + others

# This function randomly arranges the elements so we can have representation for all groups both in the training and testing set
shuffle(elements)

# Import Element from pymatgen
from pymatgen.core.periodic_table import Element as PymatgenElement

# Lists to store valid data
data_youngs_modulus = []
data_lattice_constant = []
data_melting_point = []
data_specific_heat = []
data_atomic_mass = []
data_CTE = []

valid_elements = []  # Keep track of elements that pass validation

for item in elements:

    pymat_el = PymatgenElement(item)
    mendel_el = mendel.element(item)

    # Extract properties
    young = pymat_el.youngs_modulus
    lattice = mendel_el.lattice_constant
    melt = mendel_el.melting_point
    specheat = mendel_el.specific_heat
    mass = pymat_el.atomic_mass
    cte = pymat_el.coefficient_of_linear_thermal_expansion

    # Skip element if ANY value is missing
    if None in [young, lattice, melt, specheat, mass, cte]:
        continue

    # Store valid data
    valid_elements.append(item)
    data_youngs_modulus.append(young)
    data_lattice_constant.append(lattice)
    data_melting_point.append(melt)
    data_specific_heat.append(specheat)
    data_atomic_mass.append(mass)
    data_CTE.append(cte)

print("Total valid elements:", len(valid_elements))
print("Elements used:", valid_elements)

/usr/local/lib/python3.12/dist-packages/mendeleev/models.py:397: UserWarning:

Sn has multiple allotropes, check <{self.symbol}.phase_transitions> for details.



Total valid elements: 50
Elements used: ['Co', 'Fe', 'Ir', 'Dy', 'Al', 'Nb', 'Cd', 'Mo', 'Cr', 'Ru', 'Ti', 'Re', 'Cu', 'Gd', 'Mg', 'Zr', 'Zn', 'Rh', 'Pd', 'Ta', 'Tl', 'Hf', 'Tb', 'Yb', 'Sb', 'Ce', 'Eu', 'Th', 'Bi', 'Tm', 'Sc', 'Si', 'Ho', 'Na', 'W', 'Mn', 'Li', 'Pt', 'Pb', 'Ag', 'V', 'Be', 'Sm', 'Y', 'Ni', 'Er', 'Au', 'Ba', 'Ca', 'Lu']


In [ ]:
len(elements)

51

In [ ]:
elements

['Co',
 'Fe',
 'Ir',
 'Dy',
 'Al',
 'Nb',
 'Cd',
 'Mo',
 'Cr',
 'Ru',
 'Ti',
 'Re',
 'Cu',
 'Gd',
 'Mg',
 'Zr',
 'Zn',
 'Rh',
 'Pd',
 'Ta',
 'Tl',
 'Sn',
 'Hf',
 'Tb',
 'Yb',
 'Sb',
 'Ce',
 'Eu',
 'Th',
 'Bi',
 'Tm',
 'Sc',
 'Si',
 'Ho',
 'Na',
 'W',
 'Mn',
 'Li',
 'Pt',
 'Pb',
 'Ag',
 'V',
 'Be',
 'Sm',
 'Y',
 'Ni',
 'Er',
 'Au',
 'Ba',
 'Ca',
 'Lu']

In [ ]:
data_CTE

[1.06e-05,
 1.34e-05,
 5.7e-06,
 8.8e-06,
 7.3e-06,
 1.18e-05,
 2.2e-05,
 7.1e-05,
 1.02e-05,
 2.17e-05,
 1.13e-05,
 8.2e-06,
 1.3e-05,
 4.5e-06,
 6.3e-06,
 1.65e-05,
 4.9e-06,
 2.99e-05,
 1.34e-05,
 9.9e-06,
 8.2e-06,
 9.4e-06,
 8.6e-06,
 1.33e-05,
 4.6e-05,
 2.31e-05,
 1.42e-05,
 3.08e-05,
 2.06e-05,
 1.22e-05,
 8.4e-06,
 2.23e-05,
 1.18e-05,
 1.03e-05,
 6.4e-06,
 1.1e-05,
 3.02e-05,
 5.9e-06,
 1.89e-05,
 2.6e-06,
 1.12e-05,
 2.89e-05,
 6.4e-06,
 1.1e-05,
 4.8e-06,
 2.63e-05,
 9.9e-06,
 1.27e-05,
 6.3e-06,
 3.5e-05,
 6.2e-06]

In [ ]:
print("size of data_youngs_modulus", len(data_youngs_modulus))
print("size of data_lattice_constant", len(data_lattice_constant))
print("size of data_melting_point", len(data_melting_point))
print("size of data_specific_heat", len(data_specific_heat))

size of data_youngs_modulus 50
size of data_lattice_constant 50
size of data_melting_point 50
size of data_specific_heat 50


In [ ]:
print(data_youngs_modulus)
print(data_lattice_constant)
print(data_melting_point)
print(data_specific_heat)
print(data_atomic_mass)
print(data_CTE)

[209.0, 211.0, 528.0, 61.0, 70.0, 105.0, 50.0, 329.0, 279.0, 447.0, 116.0, 463.0, 130.0, 55.0, 45.0, 68.0, 108.0, 275.0, 121.0, 186.0, 8.0, 78.0, 56.0, 24.0, 55.0, 34.0, 18.0, 79.0, 32.0, 74.0, 74.0, 47.0, 65.0, 10.0, 411.0, 198.0, 4.9, 168.0, 16.0, 83.0, 128.0, 287.0, 50.0, 64.0, 200.0, 70.0, 78.0, 13.0, 20.0, 69.0]
[2.51, 2.87, 3.84, 3.59, 4.05, 3.3, 2.98, 3.15, 2.88, 2.7, 2.95, 2.76, 3.61, 3.64, 3.21, 3.23, 2.66, 3.8, 3.89, 3.31, 3.46, 3.2, 3.6, 5.49, 4.51, 5.16, 4.61, 5.08, 4.75, 3.54, 3.31, 5.43, 3.58, 4.23, 3.16, 8.89, 3.49, 3.92, 4.95, 4.09, 3.02, 2.29, 9.0, 3.65, 3.52, 3.56, 4.08, 5.02, 5.58, 3.51]
[1768.15, 1811.15, 2719.15, 1685.15, 933.473, 2750.15, 594.219, 2895.15, 2180.15, 2606.15, 1943.15, 3458.15, 1357.77, 1586.15, 923.15, 2127.15, 692.6769999999999, 2236.15, 1827.9499999999998, 3290.15, 577.15, 2506.15, 1632.15, 1097.15, 903.778, 1072.15, 1095.15, 2023.15, 544.5519999999999, 1818.15, 1814.15, 1687.15, 1745.15, 370.94399999999996, 3687.15, 1519.15, 453.65, 2041.35, 600.

## Step 3: Data Preprocessing

In this step, the collected material properties are prepared for use in **Scikit-Learn machine learning models**.

### Key Steps
- The dataset is divided into **Training Set (45 samples)** and **Testing Set (6 samples)**.
- Training data is used to **train the ML model**, while testing data is used to **evaluate its performance**.
- Python lists are converted into **NumPy arrays**.
- The `.reshape(-1,1)` function converts data into **column vectors**, which is the format required by **Scikit-Learn regression models**.

### Properties Prepared
- Melting Point
- Young’s Modulus
- Lattice Constant
- Specific Heat
- Atomic Mass
- Coefficient of Thermal Expansion (CTE)

### Purpose
This preprocessing step ensures the data is **properly structured and ready for building linear regression models**.


In [ ]:
# -------------------------------------------------------------
# DATA PREPROCESSING FOR MACHINE LEARNING
# Goal: Prepare the dataset in the format required by Scikit-Learn
# We will build linear models to study relationships such as:
# 1. Young's Modulus vs Melting Temperature
# 2. CTE vs Melting Temperature
# 3. Lattice Constant vs Melting Temperature
# -------------------------------------------------------------

# ----------------------------
# MELTING POINT DATASET
# ----------------------------

# Split melting point data into training and testing sets
# First 45 samples → Training set
# Last 6 samples → Testing set
melt_train = data_melting_point[:45]
melt_test = data_melting_point[-6:]

# Convert lists to NumPy arrays and reshape them into column vectors
# Required format for Scikit-Learn models: [[x1], [x2], [x3], ...]
melt_train = np.array(melt_train, dtype=float).reshape(-1,1)
melt_test = np.array(melt_test, dtype=float).reshape(-1,1)


# ----------------------------
# YOUNG'S MODULUS DATASET
# ----------------------------

# Split Young's modulus data
young_train = data_youngs_modulus[:45]
young_test = data_youngs_modulus[-6:]

# Convert to column vector format
young_train = np.array(young_train, dtype=float).reshape(-1,1)
young_test = np.array(young_test, dtype=float).reshape(-1,1)


# ----------------------------
# LATTICE CONSTANT DATASET
# ----------------------------

# Split lattice constant data
lattice_train = data_lattice_constant[:45]
lattice_test = data_lattice_constant[-6:]

# Convert to column vectors
lattice_train = np.array(lattice_train, dtype=float).reshape(-1,1)
lattice_test = np.array(lattice_test, dtype=float).reshape(-1,1)


# ----------------------------
# SPECIFIC HEAT DATASET
# ----------------------------

# Split specific heat data
specheat_train = data_specific_heat[:45]
specheat_test = data_specific_heat[-6:]

# Convert to column vectors
specheat_train = np.array(specheat_train, dtype=float).reshape(-1,1)
specheat_test = np.array(specheat_test, dtype=float).reshape(-1,1)


# ----------------------------
# ATOMIC MASS DATASET
# ----------------------------

# Split atomic mass data
mass_train = data_atomic_mass[:45]
mass_test = data_atomic_mass[-6:]

# Convert to column vectors
mass_train = np.array(mass_train, dtype=float).reshape(-1,1)
mass_test = np.array(mass_test, dtype=float).reshape(-1,1)


# ----------------------------
# COEFFICIENT OF THERMAL EXPANSION (CTE)
# ----------------------------

# Split CTE data
coefTE_train = data_CTE[:45]
coefTE_test = data_CTE[-6:]

# Convert to column vectors
coefTE_train = np.array(coefTE_train, dtype=float).reshape(-1,1)
coefTE_test = np.array(coefTE_test, dtype=float).reshape(-1,1)

## Step 4: Linear Regression Model

A function is defined to train and evaluate a **Linear Regression model** using Scikit-Learn.

### Process
1. Train the model using the **training dataset**.
2. Combine training and testing data to generate predictions for the **entire dataset**.
3. Evaluate model performance using:
   - **Mean Squared Error (MSE)** – measures prediction error.
   - **R² Score** – indicates how well the model explains the data.

### Output
The function prints the **linear equation** learned by the model and returns the predicted values.

In [ ]:
# -------------------------------------------------------------
# LINEAR REGRESSION FUNCTION
# Purpose: Train a linear regression model, evaluate it,
# and generate predictions for the full dataset.
# -------------------------------------------------------------

def regression(x_train, x_test, y_train, y_test):

    # -------------------------------------------------
    # Remove rows where either X or Y contains NaN
    # -------------------------------------------------

    mask = ~np.isnan(x_train).flatten() & ~np.isnan(y_train).flatten()
    x_train = x_train[mask]
    y_train = y_train[mask]


    # ----------------------------
    # 1. Define and Train the Model
    # ----------------------------
    # Create a Linear Regression model
    model = linear_model.LinearRegression()

    # Train the model using the training dataset
    model.fit(x_train, y_train)


    # ----------------------------
    # 2. Combine Training and Testing Data
    # ----------------------------
    # Join training and testing sets so predictions
    # can be made for the entire dataset
    full_x = np.concatenate((x_train, x_test), axis=0)
    full_y = np.concatenate((y_train, y_test), axis=0)


    # ----------------------------
    # 3. Generate Predictions
    # ----------------------------
    # Use the trained model to predict outputs
    predictions = model.predict(full_x)


    # ----------------------------
    # 4. Model Evaluation Metrics
    # ----------------------------
    # Print the learned linear equation: y = mX + b
    print("Linear Equation: %.4e X + (%.4e)" % (model.coef_, model.intercept_))

    # Mean Squared Error (measures prediction error)
    print("Mean squared error: %.4e" % (mean_squared_error(full_y, predictions)))

    # R² Score (explains how well the model fits the data)
    print("Variance score: %.4f" % r2_score(full_y, predictions))


    # ----------------------------
    # 5. Return Predictions
    # ----------------------------
    return predictions

## Step 5: Data Visualization

Plotly is used to visualize the relationship between material properties and the regression model.

### What the Function Does
- Converts NumPy arrays into Python lists for plotting.
- Plots:
  - **Training Data** (green points)
  - **Testing Data** (red points)
  - **Regression Model** (blue line)
- Uses Plotly to generate **interactive graphs** for better analysis.

### Purpose
This visualization helps compare **actual data points with the predicted regression line**, making it easier to evaluate how well the model fits the dataset.

In [ ]:
# -------------------------------------------------------------
# PLOTLY VISUALIZATION SETUP
# -------------------------------------------------------------
# Import Plotly graph objects and configure the renderer
# so plots display properly inside Google Colab notebooks

import plotly.io as pio
import plotly.graph_objects as go

# Set renderer for Colab environment
pio.renderers.default = "colab"


# -------------------------------------------------------------
# PLOTTING FUNCTION
# Purpose:
# Visualize training data, testing data, and the regression
# model prediction line on a single interactive plot.
#
# Parameters:
# x_train, x_test : Feature values used for training/testing
# y_train, y_test : Target values for training/testing
# x_label         : Label for x-axis
# y_label         : Label for y-axis
# predictions     : Model predicted values
# -------------------------------------------------------------

def plot(x_train, x_test, y_train, y_test, x_label, y_label, predictions):

    # ---------------------------------------------------------
    # 1. Convert NumPy column vectors to Python lists
    #
    # Example transformation:
    #
    # [[x],
    #  [y],
    #  [z]]
    #
    # becomes:
    #
    # [x, y, z]
    #
    # Plotly works best with simple Python lists
    # ---------------------------------------------------------

    x_train = x_train.reshape(1, -1).tolist()[0]
    x_test = x_test.reshape(1, -1).tolist()[0]
    y_train = y_train.reshape(1, -1).tolist()[0]
    y_test = y_test.reshape(1, -1).tolist()[0]
    predictions = predictions.reshape(1, -1).tolist()[0]

    # Combine training and testing x values
    # This allows us to draw one continuous regression line
    full_x_list = x_train + x_test


    # ---------------------------------------------------------
    # 2. Define Plot Layout
    #
    # Controls figure size, axis labels, grid lines,
    # legend styling, and hover behavior.
    # ---------------------------------------------------------

    layout0 = go.Layout(

        hovermode='closest',      # Show closest data point when hovering
        width=800,                # Plot width
        height=600,               # Plot height
        showlegend=True,          # Display legend

        # X-axis configuration
        xaxis=dict(
            title=go.layout.xaxis.Title(
                text=x_label,
                font=dict(size=24)
            ),
            zeroline=False,       # Remove axis zero line
            gridwidth=1,          # Grid line thickness
            tickfont=dict(size=18)
        ),

        # Y-axis configuration
        yaxis=dict(
            title=go.layout.yaxis.Title(
                text=y_label,
                font=dict(size=24)
            ),
            zeroline=False,
            gridwidth=1,
            tickfont=dict(size=18)
        ),

        # Legend style
        legend=dict(font=dict(size=20))
    )


    # ---------------------------------------------------------
    # 3. Training Data Trace
    #
    # Green markers represent data used to train the model
    # ---------------------------------------------------------

    training = go.Scatter(
        x=x_train,
        y=y_train,
        mode='markers',
        marker=dict(size=10, color='green'),
        name="Training Data"
    )


    # ---------------------------------------------------------
    # 4. Testing Data Trace
    #
    # Red markers represent unseen data used for evaluating
    # model performance
    # ---------------------------------------------------------

    testing = go.Scatter(
        x=x_test,
        y=y_test,
        mode='markers',
        marker=dict(size=10, color='red'),
        name="Testing Data"
    )


    # ---------------------------------------------------------
    # 5. Regression Model Trace
    #
    # Blue line shows the relationship predicted by the model
    # ---------------------------------------------------------

    prediction_line = go.Scatter(
        x=full_x_list,
        y=predictions,
        mode='lines',
        line=dict(color="blue", width=2),
        name="Regression Model"
    )


    # ---------------------------------------------------------
    # 6. Combine all traces and create the figure
    # ---------------------------------------------------------

    data = [training, testing, prediction_line]

    fig = go.Figure(data=data, layout=layout0)


    # ---------------------------------------------------------
    # 7. Display interactive plot
    # ---------------------------------------------------------

    fig.show()

## Step 6: Running the Model and Visualization

1. The `regression()` function is called to **train the linear regression model** using melting temperature to predict **Young's Modulus**.
2. The model returns **predicted values** for the dataset.
3. The `plot()` function visualizes:
   - Training data
   - Testing data
   - The regression line fitted by the model

This step helps analyze the **relationship between melting temperature and material stiffness**.

In [ ]:
# -------------------------------------------------------------
# RUN THE REGRESSION MODEL
# -------------------------------------------------------------

# Call the regression() function
# Input:
#   melt_train, melt_test  → Melting temperature (independent variable X)
#   young_train, young_test → Young's modulus (dependent variable Y)
# Output:
#   predictions → Model-predicted values of Young's modulus

predictions = regression(melt_train, melt_test, young_train, young_test)


# -------------------------------------------------------------
# VISUALIZE THE RESULTS
# -------------------------------------------------------------

# Call the plotting function to visualize:
# 1. Training data (green points)
# 2. Testing data (red points)
# 3. Regression model fit (blue line)

plot(
    melt_train,
    melt_test,
    young_train,
    young_test,
    "Melting Temperature (K)",   # X-axis label
    "Young's Modulus (GPa)",     # Y-axis label
    predictions                  # Predicted values from the model
)

Linear Equation: 1.1835e-01 X + (-6.4983e+01)
Mean squared error: 7.4920e+03
Variance score: 0.5247


/tmp/ipykernel_231/2506083455.py:48: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)



### Model Prediction and Visualization

In this step, we use the trained regression model to predict the **lattice constant** from the **melting temperature**.

1. The `regression()` function trains a linear regression model using the training data.
2. It then generates predictions for both the training and testing sets.
3. The `plot()` function visualizes:
   - Training data (green points)
   - Testing data (red points)
   - The regression model fit (blue line)

This visualization helps us understand how well the melting temperature predicts the lattice constant.

In [ ]:
# -------------------------------------------------------------
# Generate Predictions using the Regression Model
# -------------------------------------------------------------
# The regression() function trains a linear regression model
# using the training dataset and returns predictions for the
# combined training and testing feature values.

predictions = regression(
    melt_train,      # Training feature: Melting Temperature
    melt_test,       # Testing feature: Melting Temperature
    lattice_train,   # Training target: Lattice Constant
    lattice_test     # Testing target: Lattice Constant
)


# -------------------------------------------------------------
# Visualize Model Performance
# -------------------------------------------------------------
# The plot() function creates an interactive Plotly graph that
# shows:
#  - Training data (green markers)
#  - Testing data (red markers)
#  - Regression model fit (blue line)

plot(
    melt_train,      # Training feature values
    melt_test,       # Testing feature values
    lattice_train,   # Training target values
    lattice_test,    # Testing target values
    "Melting Temperature (K)",   # X-axis label
    "Lattice Constant (Å)",      # Y-axis label
    predictions      # Model predictions
)

Linear Equation: -4.5620e-04 X + (4.6605e+00)
Mean squared error: 1.5054e+00
Variance score: 0.0890


/tmp/ipykernel_231/2506083455.py:48: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)



### Predicting the Coefficient of Linear Thermal Expansion

In this section, we investigate the relationship between **melting temperature** and the **coefficient of linear thermal expansion (CTE)**.

1. A **linear regression model** is trained using the training dataset.
2. The model predicts the **CTE values** based on melting temperature.
3. The `plot()` function visualizes:
   - **Training data** (green points)
   - **Testing data** (red points)
   - **Regression model fit** (blue line)

The coefficient of linear thermal expansion measures how much a material expands when the temperature changes. Studying its relationship with melting temperature can help identify trends in the thermal behavior of different elements.

In [ ]:
# -------------------------------------------------------------
# Generate Predictions using the Regression Model
# -------------------------------------------------------------
# The regression() function trains a linear regression model
# using melting temperature as the input feature and the
# coefficient of linear thermal expansion as the target.

predictions = regression(
    melt_train,        # Training feature: Melting Temperature
    melt_test,         # Testing feature: Melting Temperature
    coefTE_train,      # Training target: Coefficient of Thermal Expansion
    coefTE_test        # Testing target: Coefficient of Thermal Expansion
)


# -------------------------------------------------------------
# Visualize Model Predictions
# -------------------------------------------------------------
# The plot() function generates an interactive Plotly graph
# showing the relationship between melting temperature and CTE.
#
# Plot components:
# - Green markers → Training dataset
# - Red markers → Testing dataset
# - Blue line → Regression model prediction

plot(
    melt_train,                     # Training feature values
    melt_test,                      # Testing feature values
    coefTE_train,                   # Training target values
    coefTE_test,                    # Testing target values
    "Melting Temperature (K)",      # X-axis label
    "Coefficient of Linear Thermal Expansion (K<sup>-1</sup>)",  # Y-axis label
    predictions                     # Model predictions
)

Linear Equation: -1.0630e-08 X + (3.2989e-05)
Mean squared error: 7.7402e-11
Variance score: 0.4578


/tmp/ipykernel_231/2506083455.py:48: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)



## Final Conclusions

### 1. Young's Modulus vs. Melting Temperature
- **Result:** Strong positive correlation.  
- **Meaning:** Materials that require more heat to melt usually have **stronger atomic bonds**.  
- **Implication:** Strong bonds make the material **stiffer**, resulting in a higher Young’s Modulus.  
- **Observation:** Testing data points lie close to the regression line, showing the model gives **reliable predictions**.

---

### 2. Lattice Constant vs. Melting Temperature
- **Result:** Weak negative relationship with high scatter.  
- **Meaning:** The **size of the atomic lattice** does not strongly determine melting temperature.  
- **Implication:** Large atoms often have weaker bonds, but this is **not a strict rule**.  
- **Observation:** High data variation indicates that **lattice geometry alone is a poor predictor** of melting behavior.

---

### 3. Coefficient of Thermal Expansion vs. Melting Temperature
- **Result:** Clear negative correlation.  
- **Meaning:** Materials with **higher melting points expand less when heated**.  
- **Implication:** Strong atomic bonds restrict atomic vibrations and expansion.  
- **Observation:** The linear model predicts slightly negative values at high temperatures, suggesting that a **non-linear (polynomial) model may improve accuracy**.

---

### Overall Insight
- **Bond strength plays a key role in determining material properties.**
- **Melting temperature is a good predictor of mechanical stiffness**, but not of lattice geometry.
- **Thermal expansion decreases as bond strength increases.**